# Compute outputs of GeoCLIP model on images and filters.

This data is used to quantify the effects of applying aesthetic filters to images.

In [ ]:
from pathlib import Path
import os
from tqdm import tqdm
from glob import glob
import pandas as pd
from geoclip import GeoCLIP
import torch

## Load model

In [ ]:
model = GeoCLIP()
model.to("cuda")

## Setup

In [ ]:
os.makedirs("./csv_files", exist_ok=True)

**Note**: Update this path accordingly

In [ ]:
base_dir = "path_to_filtered_dataset"  # Update this path accordingly

In [ ]:
filter_names = os.listdir(base_dir)
print(filter_names)

## Run the model on different images with filters applied

In [ ]:
for filter_name in filter_names:
    print(f"Processing {filter_name}")

    # List all images
    all_images = glob(f"{os.path.join(base_dir, filter_name)}/*.jpg")

    # Store image ID and predicted latitude and longitude
    all_image_ids = []
    all_coords = []
    for i in tqdm(range(len(all_images))):
        image_path = all_images[i]
        score_key = Path(image_path).stem
        with torch.no_grad():
            top_pred_gps, top_pred_prob = model.predict(image_path, top_k=1)
            lat, lon = top_pred_gps[0]
            lat = str(float(lat.item()))
            lon = str(float(lon.item()))
            predicted_loc = (lat, lon)
            all_image_ids.append(score_key)
            all_coords.append(predicted_loc)

        # Clear cache and take backups periodically
        if i % 1000 == 0:
            torch.cuda.empty_cache()
            df = pd.DataFrame({"image_id": all_image_ids, "comp_coords": all_coords})
            df.to_csv(f"geoclip_{filter_name}.csv", index=False)

    # Save results to CSV file
    df = pd.DataFrame({"image_id": all_image_ids, "comp_coords": all_coords})
    df.to_csv(f"./csv_files/geoclip_{filter_name}.csv", index=False)